# AffectLab — MELD text emotion training

This notebook fine-tunes `microsoft/deberta-v3-small` on MELD's official text splits, evaluates three random seeds, writes artifacts to Google Drive, and exports an AffectLab-compatible label mapping. Run cells in order with a Colab GPU runtime. Do not place dataset or Hugging Face credentials in Git.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
print('Google Drive mounted.')

In [ ]:
!nvidia-smi
import torch

assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU before training'
print(torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/ralucabiras/emotion-aware-role-play-model.git'
REPO_DIR = Path('/content/emotion-aware-role-play-model')
if not REPO_DIR.exists():
    !git clone $REPO_URL $REPO_DIR
%cd /content/emotion-aware-role-play-model
!git pull --ff-only
!python -m pip install -q -r requirements-ml.txt
from google.colab import userdata  # noqa: E402
from huggingface_hub import login  # noqa: E402

hf_token = userdata.get('HF_TOKEN')
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
else:
    print('HF_TOKEN is not set in Colab Secrets; public downloads can still work.')

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/AffectLab')
DATA_DIR = DRIVE_ROOT / 'data' / 'meld-text-v1'
RUN_DIR = DRIVE_ROOT / 'runs' / 'meld-deberta-v3-small'
DATA_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('Data:', DATA_DIR)
print('Runs:', RUN_DIR)

In [ ]:
!python -m ml.preprocessing.meld --output-dir "$DATA_DIR"
import json

manifest = json.loads((DATA_DIR / 'manifest.json').read_text())
manifest['splits']

## Train
Start with seed 42 to validate the pipeline. Set `RUN_ALL_SEEDS = True` only after that run completes and its metrics/artifacts look correct. Three seeds are needed for the dissertation result.

In [ ]:
RUN_ALL_SEEDS = False
seeds = [13, 42, 73] if RUN_ALL_SEEDS else [42]
for seed in seeds:
    print(f'\n=== Training seed {seed} ===')
    !python -m ml.training.train_meld --config configs/meld_deberta_v3_small.json --data-dir "$DATA_DIR" --output-dir "$RUN_DIR" --seed $seed

In [ ]:
if RUN_ALL_SEEDS:
    !python -m ml.evaluation.summarize_runs "$RUN_DIR"
    summary = json.loads((RUN_DIR / 'summary.json').read_text())
    display(summary)
else:
    metrics = json.loads((RUN_DIR / 'seed-42' / 'metrics.json').read_text())
    display(metrics['test_metrics'])
    display(metrics['test_per_class'])

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(RUN_DIR / 'seed-42' / 'confusion_matrix.png')))

In [ ]:
MODEL_DIR = RUN_DIR / 'seed-42' / 'model'
!python -m ml.training.smoke_inference "$MODEL_DIR" "I am worried my manager will think I failed" "I am proud that I set a boundary" "I cannot believe that happened"

## Completion checklist

- Confirm `metrics.json`, `test_predictions.jsonl`, and `confusion_matrix.png` exist for each seed.
- Confirm the model folder contains `affectlab_metadata.json`.
- Do not select a seed because it has the best test score; report mean and standard deviation across the predeclared seeds.
- Preserve the Drive run directory unchanged for dissertation reproducibility.
- Do not publish or upload checkpoints until dataset/model licensing has been reviewed for that destination.